In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Setup - Ensuring 5,000 high-quality rows
np.random.seed(42)
n_rows = 5000
countries = ['Germany', 'Netherlands', 'Latvia', 'Estonia', 'Lithuania', 'Cyprus', 'Malta', 'Luxembourg']

# 2. Base Data Generation
data = {
    'transaction_id': [f'TXN{i:04d}' for i in range(n_rows)],
    'timestamp': [datetime(2023, 1, 1) + timedelta(minutes=i*15) for i in range(n_rows)],
    'sender_id': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n_rows)],
    'receiver_id': [f'CUST{np.random.randint(2001, 3000)}' for _ in range(n_rows)],
    'sender_country': np.random.choice(countries, n_rows),
    'receiver_country': np.random.choice(countries, n_rows),
    'amount_eur': np.random.uniform(500, 15000, n_rows),
    'is_flagged': [False] * n_rows,
    'flag_reason': ['Clean'] * n_rows,
    'risk_score': np.random.uniform(5, 35, n_rows) # Low base risk for clean txns
}

df = pd.DataFrame(data)

# 3. Inject "Structuring" (The €10k Spike - Crucial for Worksheet 7)
# We force a cluster of transactions just below the limit
struct_idx = df.sample(250).index
df.loc[struct_idx, 'amount_eur'] = np.random.uniform(9600, 9950, 250)
df.loc[struct_idx, 'is_flagged'] = True
df.loc[struct_idx, 'flag_reason'] = 'Structuring'
df.loc[struct_idx, 'risk_score'] = np.random.uniform(75, 98, 250)

# 4. Inject "Layering" (High Risk Pairs - Crucial for Worksheet 10)
# Rapid flow between specific countries (e.g., Cyprus to Malta)
layer_idx = df[~df.index.isin(struct_idx)].sample(150).index
df.loc[layer_idx, 'sender_country'] = 'Cyprus'
df.loc[layer_idx, 'receiver_country'] = 'Malta'
df.loc[layer_idx, 'is_flagged'] = True
df.loc[layer_idx, 'flag_reason'] = 'Layering'
df.loc[layer_idx, 'risk_score'] = np.random.uniform(65, 90, 150)

# 5. Final Formatting
df['risk_score'] = df['risk_score'].round(2)
df['amount_eur'] = df['amount_eur'].round(2)

# Save the file
df.to_csv('aml_final_blueprint.csv', index=False)
print("SUCCESS: 'aml_final_blueprint.csv' generated.")

SUCCESS: 'aml_final_blueprint.csv' generated.
